# Results

In [1]:
# ! path to imagenet val dataset
dataset_dir = '/media/datasets/imagenet/val'

In [2]:
import sys
import os

base_dir = os.path.abspath('..')
if base_dir not in sys.path:
    sys.path.append(base_dir)

sret_weights_path = os.path.join(base_dir, 'weights/SReT_T_distill.pth')

import torch
import timm
import tome
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import eval_cpu
from eval_gpu import evaluate
from sret.SReT import SReT_T_distill
import integration.PiT_ToMe
import integration.SReT_ToMe

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Current GPU Device: {torch.cuda.current_device()}")

batch_size = 128 # batch size doesn't matter since it's only used for acc evaluations
dataset_transform = transforms.Compose([
    transforms.Resize(256), 
    transforms.CenterCrop(224), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
dataset = datasets.ImageFolder(dataset_dir, transform=dataset_transform)
dataset_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True) 
print(f'Loaded {len(dataset)} ImageNet-1k Validation images')

CUDA available: True
GPU Device Name: NVIDIA GeForce RTX 4060 Ti
Current GPU Device: 0
Loaded 50000 ImageNet-1k Validation images


#### DeiT-Tiny-Distill 

In [4]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.40 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          2.17 G
Throughput (BS=128):            1795.64 images/sec
Throughput (BS=64):             1916.04 images/sec
Throughput (BS=32):             1997.31 images/sec
Throughput (BS=16):             2320.26 images/sec
Throughput (BS=1):               728.13 images/sec
Peak Activation Memory (BS=128):         227.20 MB
Peak Activation Memory (BS=64):          113.03 MB
Peak Activation Memory (BS=32):           56.44 MB
Peak Activation Memory (BS=16):           28.59 MB
Peak Activation Memory (BS=1):             1.76 MB



In [3]:
_ = eval_cpu.evaluate("deit")

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.21 ms
Throughput:                         138.74 img/sec



#### DeiT-Tiny-Distill + ToMe | Constant Reduction

In [4]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
tome.patch.timm(model, prop_attn=True)
model = model.cuda().eval()
model.r = 10
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.47 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          1.51 G
Throughput (BS=128):            2432.47 images/sec
Throughput (BS=64):             2646.35 images/sec
Throughput (BS=32):             2679.56 images/sec
Throughput (BS=16):             2643.10 images/sec
Throughput (BS=1):               244.47 images/sec
Peak Activation Memory (BS=128):         232.62 MB
Peak Activation Memory (BS=64):          117.20 MB
Peak Activation Memory (BS=32):           58.93 MB
Peak Activation Memory (BS=16):           29.64 MB
Peak Activation Memory (BS=1):             1.82 MB



In [5]:
_ = eval_cpu.evaluate("deit+tome+c", constant_r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.25 ms
Throughput:                         137.88 img/sec



In [6]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
tome.patch.timm(model, prop_attn=True)
model = model.cuda().eval()
model.r = 20
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            61.49 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          0.92 G
Throughput (BS=128):            4002.00 images/sec
Throughput (BS=64):             4213.01 images/sec
Throughput (BS=32):             4157.48 images/sec
Throughput (BS=16):             3662.68 images/sec
Throughput (BS=1):               249.06 images/sec
Peak Activation Memory (BS=128):         216.16 MB
Peak Activation Memory (BS=64):          110.27 MB
Peak Activation Memory (BS=32):           55.12 MB
Peak Activation Memory (BS=16):           27.62 MB
Peak Activation Memory (BS=1):             1.69 MB



In [7]:
_ = eval_cpu.evaluate("deit+tome+c", constant_r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   5.48 ms
Throughput:                         182.52 img/sec



#### PiT-Tiny-Distill

In [8]:
model = timm.create_model("pit_ti_distilled_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.42 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          1.01 G
Throughput (BS=128):            1789.10 images/sec
Throughput (BS=64):             1840.51 images/sec
Throughput (BS=32):             1929.72 images/sec
Throughput (BS=16):             2064.57 images/sec
Throughput (BS=1):               671.18 images/sec
Peak Activation Memory (BS=128):        1203.36 MB
Peak Activation Memory (BS=64):          603.73 MB
Peak Activation Memory (BS=32):          302.11 MB
Peak Activation Memory (BS=16):          151.92 MB
Peak Activation Memory (BS=1):             9.40 MB



In [9]:
_ = eval_cpu.evaluate("pit")

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   6.18 ms
Throughput:                         161.88 img/sec



#### PiT-Tiny-Distill + ToMe | Constant Reduction

In [10]:
model = integration.PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="constant", constant_r=10)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.76 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.80 G
Throughput (BS=128):            1558.92 images/sec
Throughput (BS=64):             1608.96 images/sec
Throughput (BS=32):             1648.83 images/sec
Throughput (BS=16):             1645.95 images/sec
Throughput (BS=1):               223.32 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [11]:
_ = eval_cpu.evaluate("pit+tome+c", constant_r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   8.32 ms
Throughput:                         120.24 img/sec



In [12]:
model = integration.PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="constant", constant_r=20)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            71.10 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.66 G
Throughput (BS=128):            1719.98 images/sec
Throughput (BS=64):             1780.22 images/sec
Throughput (BS=32):             1803.98 images/sec
Throughput (BS=16):             1754.63 images/sec
Throughput (BS=1):               221.67 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [13]:
_ = eval_cpu.evaluate("pit+tome+c", constant_r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.70 ms
Throughput:                         129.79 img/sec



#### PiT-Tiny-Distill + ToMe | Linear Reduction

In [14]:
model = integration.PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="linear", linear_r=10)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.06 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.86 G
Throughput (BS=128):            1573.34 images/sec
Throughput (BS=64):             1630.12 images/sec
Throughput (BS=32):             1673.72 images/sec
Throughput (BS=16):             1660.31 images/sec
Throughput (BS=1):               234.55 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [15]:
_ = eval_cpu.evaluate("pit+tome+l", linear_r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   8.22 ms
Throughput:                         121.62 img/sec



In [16]:
model = integration.PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="linear", linear_r=20)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            71.13 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.71 G
Throughput (BS=128):            1788.91 images/sec
Throughput (BS=64):             1851.27 images/sec
Throughput (BS=32):             1859.41 images/sec
Throughput (BS=16):             1788.28 images/sec
Throughput (BS=1):               233.91 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [17]:
_ = eval_cpu.evaluate("pit+tome+l", linear_r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.64 ms
Throughput:                         130.93 img/sec



#### PiT-Tiny-Distill + ToMe | Exponential Reduction

In [18]:
model = integration.PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="exponential", initial_r=0.25, alpha=0.0)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            72.03 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.80 G
Throughput (BS=128):            1902.01 images/sec
Throughput (BS=64):             1999.65 images/sec
Throughput (BS=32):             2067.72 images/sec
Throughput (BS=16):             2080.14 images/sec
Throughput (BS=1):               378.28 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [19]:
_ = eval_cpu.evaluate("pit+tome+e", initial_r=0.25, alpha=0)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   6.77 ms
Throughput:                         147.75 img/sec



In [20]:
model = integration.PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="exponential", initial_r=0.40, alpha=0.2)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            62.63 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.63 G
Throughput (BS=128):            2328.09 images/sec
Throughput (BS=64):             2435.98 images/sec
Throughput (BS=32):             2443.32 images/sec
Throughput (BS=16):             2342.46 images/sec
Throughput (BS=1):               284.88 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [21]:
_ = eval_cpu.evaluate("pit+tome+e", initial_r=0.40, alpha=0.2)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   6.62 ms
Throughput:                         151.06 img/sec



In [22]:
model = integration.PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="exponential", initial_r=0.10, alpha=0.6)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.90 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.88 G
Throughput (BS=128):            1606.83 images/sec
Throughput (BS=64):             1653.23 images/sec
Throughput (BS=32):             1710.59 images/sec
Throughput (BS=16):             1710.81 images/sec
Throughput (BS=1):               230.54 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [23]:
_ = eval_cpu.evaluate("pit+tome+e", initial_r=0.1, alpha=0.6)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   8.25 ms
Throughput:                         121.19 img/sec



#### SReT-Tiny-Distill

In [24]:
model = SReT_T_distill(pretrained=False)
checkpoint = torch.load(sret_weights_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            77.40 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.91 G
Throughput (BS=128):            1067.92 images/sec
Throughput (BS=64):             1164.30 images/sec
Throughput (BS=32):             1246.04 images/sec
Throughput (BS=16):             1295.99 images/sec
Throughput (BS=1):               220.74 images/sec
Peak Activation Memory (BS=128):         795.76 MB
Peak Activation Memory (BS=64):          397.88 MB
Peak Activation Memory (BS=32):          200.88 MB
Peak Activation Memory (BS=16):          100.44 MB
Peak Activation Memory (BS=1):             6.22 MB



In [25]:
_ = eval_cpu.evaluate("sret")

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  11.88 ms
Throughput:                          84.19 img/sec



#### SReT-Tiny-Distill + ToMe | Constant Reduction

In [26]:
model = integration.SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="constant", constant_r=10)
checkpoint = torch.load(sret_weights_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            70.84 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.32 G
Throughput (BS=128):            1171.58 images/sec
Throughput (BS=64):             1262.99 images/sec
Throughput (BS=32):             1312.38 images/sec
Throughput (BS=16):             1162.85 images/sec
Throughput (BS=1):               111.19 images/sec
Peak Activation Memory (BS=128):         770.04 MB
Peak Activation Memory (BS=64):          385.13 MB
Peak Activation Memory (BS=32):          192.70 MB
Peak Activation Memory (BS=16):           96.22 MB
Peak Activation Memory (BS=1):             6.01 MB



In [27]:
_ = eval_cpu.evaluate("sret+tome+c", constant_r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  13.31 ms
Throughput:                          75.12 img/sec



In [28]:
model = integration.SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="constant", constant_r=20)
checkpoint = torch.load(sret_weights_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            41.13 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.06 G
Throughput (BS=128):            1326.51 images/sec
Throughput (BS=64):             1424.84 images/sec
Throughput (BS=32):             1459.49 images/sec
Throughput (BS=16):             1164.65 images/sec
Throughput (BS=1):               110.64 images/sec
Peak Activation Memory (BS=128):         756.22 MB
Peak Activation Memory (BS=64):          380.08 MB
Peak Activation Memory (BS=32):          189.43 MB
Peak Activation Memory (BS=16):           96.02 MB
Peak Activation Memory (BS=1):             5.91 MB



In [29]:
_ = eval_cpu.evaluate("sret+tome+c", constant_r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  12.08 ms
Throughput:                          82.78 img/sec



#### SReT-Tiny-Distill + ToMe | Linear Reduction

In [32]:
model = integration.SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="linear", linear_r=10)
checkpoint = torch.load(sret_weights_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.72 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.46 G
Throughput (BS=128):            1171.23 images/sec
Throughput (BS=64):             1264.72 images/sec
Throughput (BS=32):             1309.28 images/sec
Throughput (BS=16):             1184.07 images/sec
Throughput (BS=1):               112.65 images/sec
Peak Activation Memory (BS=128):         756.22 MB
Peak Activation Memory (BS=64):          380.08 MB
Peak Activation Memory (BS=32):          189.43 MB
Peak Activation Memory (BS=16):           96.02 MB
Peak Activation Memory (BS=1):             5.91 MB



In [33]:
_ = eval_cpu.evaluate("sret+tome+l", linear_r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  13.92 ms
Throughput:                          71.81 img/sec



In [34]:
model = integration.SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="linear", linear_r=20)
checkpoint = torch.load(sret_weights_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            14.82 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.07 G
Throughput (BS=128):            1423.35 images/sec
Throughput (BS=64):             1520.79 images/sec
Throughput (BS=32):             1541.85 images/sec
Throughput (BS=16):             1225.78 images/sec
Throughput (BS=1):               116.01 images/sec
Peak Activation Memory (BS=128):         729.71 MB
Peak Activation Memory (BS=64):          366.70 MB
Peak Activation Memory (BS=32):          183.85 MB
Peak Activation Memory (BS=16):           91.61 MB
Peak Activation Memory (BS=1):             5.70 MB



In [35]:
_ = eval_cpu.evaluate("sret+tome+l", linear_r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  12.02 ms
Throughput:                          83.20 img/sec



#### SReT-Tiny-Distill + ToMe | Exponential Reduction

In [36]:
model = integration.SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="exponential", initial_r=0.25, alpha=0.0)
checkpoint = torch.load(sret_weights_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            75.91 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.49 G
Throughput (BS=128):            1356.32 images/sec
Throughput (BS=64):             1483.31 images/sec
Throughput (BS=32):             1565.36 images/sec
Throughput (BS=16):             1573.36 images/sec
Throughput (BS=1):               171.73 images/sec
Peak Activation Memory (BS=128):         489.38 MB
Peak Activation Memory (BS=64):          245.46 MB
Peak Activation Memory (BS=32):          122.71 MB
Peak Activation Memory (BS=16):           62.09 MB
Peak Activation Memory (BS=1):             3.90 MB



In [37]:
_ = eval_cpu.evaluate("sret+tome+e", initial_r=0.25, alpha=0)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  11.37 ms
Throughput:                          87.98 img/sec



In [38]:
model = integration.SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="exponential", initial_r=0.40, alpha=0.2)
checkpoint = torch.load(sret_weights_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            69.83 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.13 G
Throughput (BS=128):            1870.01 images/sec
Throughput (BS=64):             2011.81 images/sec
Throughput (BS=32):             2057.28 images/sec
Throughput (BS=16):             1812.43 images/sec
Throughput (BS=1):               141.07 images/sec
Peak Activation Memory (BS=128):         391.51 MB
Peak Activation Memory (BS=64):          195.76 MB
Peak Activation Memory (BS=32):           99.88 MB
Peak Activation Memory (BS=16):           48.94 MB
Peak Activation Memory (BS=1):             3.90 MB



In [39]:
_ = eval_cpu.evaluate("sret+tome+e", initial_r=0.40, alpha=0.2)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  11.03 ms
Throughput:                          90.64 img/sec



In [40]:
model = integration.SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="exponential", initial_r=0.1, alpha=0.6)
checkpoint = torch.load(sret_weights_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            76.38 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.61 G
Throughput (BS=128):            1170.23 images/sec
Throughput (BS=64):             1274.09 images/sec
Throughput (BS=32):             1342.01 images/sec
Throughput (BS=16):             1324.39 images/sec
Throughput (BS=1):               128.22 images/sec
Peak Activation Memory (BS=128):         665.63 MB
Peak Activation Memory (BS=64):          335.25 MB
Peak Activation Memory (BS=32):          167.06 MB
Peak Activation Memory (BS=16):           83.09 MB
Peak Activation Memory (BS=1):             5.19 MB



In [41]:
_ = eval_cpu.evaluate("sret+tome+e", initial_r=0.10, alpha=0.6)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  13.57 ms
Throughput:                          73.68 img/sec

